# Attention Mechanisms From Scratch

Wiki reference for [attention mechanisms](https://ml-viz-ruby.vercel.app/wiki/attention-mechanisms).

**The idea in one sentence.** Scaled dot-product attention turns each query into a
**probability-weighted average** of value vectors — the $1/\sqrt{d_k}$ scaling keeps softmax
out of saturation, a **causal mask** blocks attending to the future, and **multi-head / grouped-query**
variants trade off expressiveness against KV-cache memory.

We implement scaled dot-product attention, causal masking, and MHA/GQA from scratch (PyTorch),
**validate that weights form a distribution, that masking blocks the future, and that GQA
shrinks the KV cache**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
plt.style.use('dark_background')
torch.manual_seed(0)

## 1 — Scaled dot-product attention (from scratch)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Q, K: (..., n, d_k)  V: (..., n, d_v)"""
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / d_k**0.5       # (..., n_q, n_kv)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

# 3-token worked example from the wiki
Q = torch.tensor([[2.,0],[0,2],[2,2]])
K = torch.tensor([[2.,0],[0,2],[1,1]])
V = torch.tensor([[1.,0],[0,1],[10,10]])

out, W = scaled_dot_product_attention(Q, K, V)
print('Attention weights (rows sum to 1):')
print(W.round(decimals=3))
print('Output:')
print(out.round(decimals=3))

### Validate: attention weights form a distribution

Softmax over the scaled scores makes each query's attention weights a **probability
distribution** — non-negative and summing to 1 — so the output is a convex combination of the
value vectors. We confirm each row sums to 1.

In [ ]:
print('row sums:', W.sum(dim=-1).round(decimals=4).tolist())
assert torch.allclose(W.sum(dim=-1), torch.ones(W.shape[0]), atol=1e-5), 'each query attends with a probability distribution'
assert (W >= 0).all(), 'attention weights are non-negative'
print('\n✅ attention = a softmax-weighted average of values (rows are distributions)')

## 2 — Causal (masked) self-attention

In [ ]:
n = 5
# Lower-triangular mask: True = masked (should not attend)
causal_mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
print('Causal mask (True = blocked):')
print(causal_mask.int())

X = torch.randn(1, n, 8)  # (batch=1, seq=5, d=8)
W_q = torch.randn(8, 4);  W_k = torch.randn(8, 4);  W_v = torch.randn(8, 4)
Q_ = X @ W_q;  K_ = X @ W_k;  V_ = X @ W_v

out_causal, W_causal = scaled_dot_product_attention(Q_, K_, V_, mask=causal_mask)
print('Weights (upper triangle should be 0):')
print(W_causal[0].round(decimals=3))

### Validate: the causal mask blocks the future

A decoder must not attend to positions after the current token. The upper-triangular mask sets
those scores to $-\infty$, so after softmax every **future** weight is exactly 0 while each row
still sums to 1. We confirm.

In [ ]:
W0 = W_causal[0]
upper = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
print('future (upper-triangle) weights all zero:', bool((W0[upper] == 0).all()))
assert (W0[upper] == 0).all(), 'causal masking zeros every future attention weight'
assert torch.allclose(W0.sum(dim=-1), torch.ones(n), atol=1e-5), 'each row is still a valid distribution'
print('\n✅ causal masking enforces autoregressive attention — no peeking ahead')

## 3 — Multi-Head Attention (MHA)

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h = h
        self.d_k = d_model // h
        self.W_q = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_k = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_v = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_o = torch.nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, n, _ = x.shape
        def project(W):
            return W(x).view(B, n, self.h, self.d_k).transpose(1, 2)  # (B, h, n, d_k)
        Q, K, V = project(self.W_q), project(self.W_k), project(self.W_v)
        out, _ = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, n, -1)
        return self.W_o(out)

d_model, h, n = 64, 8, 10
mha = MultiHeadAttention(d_model, h)
x = torch.randn(2, n, d_model)
out_mha = mha(x)
print(f'MHA output shape: {out_mha.shape}')  # (2, 10, 64)
print(f'Parameters: {sum(p.numel() for p in mha.parameters()):,}')

## 4 — Grouped-Query Attention (GQA)

GQA shares K/V heads across groups of query heads. MHA is $G=h$, MQA is $G=1$.

In [ ]:
class GroupedQueryAttention(torch.nn.Module):
    def __init__(self, d_model, h, G):
        super().__init__()
        assert h % G == 0
        self.h, self.G = h, G
        self.d_k = d_model // h
        self.W_q = torch.nn.Linear(d_model, d_model, bias=False)          # h query heads
        self.W_k = torch.nn.Linear(d_model, G * self.d_k, bias=False)     # G key heads
        self.W_v = torch.nn.Linear(d_model, G * self.d_k, bias=False)     # G value heads
        self.W_o = torch.nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, n, _ = x.shape
        Q = self.W_q(x).view(B, n, self.h, self.d_k).transpose(1, 2)      # (B, h, n, d_k)
        K = self.W_k(x).view(B, n, self.G, self.d_k).transpose(1, 2)      # (B, G, n, d_k)
        V = self.W_v(x).view(B, n, self.G, self.d_k).transpose(1, 2)
        # Expand K, V to match h query heads: each group of h//G queries shares one K,V
        reps = self.h // self.G
        K = K.repeat_interleave(reps, dim=1)   # (B, h, n, d_k)
        V = V.repeat_interleave(reps, dim=1)
        out, _ = scaled_dot_product_attention(Q, K, V)
        out = out.transpose(1, 2).contiguous().view(B, n, -1)
        return self.W_o(out)

# Compare parameter counts and KV cache across variants
d_model = 512;  h = 16;  n_kv = 512  # sequence length for cache calc
for G in [16, 4, 1]:
    gqa = GroupedQueryAttention(d_model, h, G)
    params = sum(p.numel() for p in gqa.parameters())
    kv_cache_elements = 2 * n_kv * G * (d_model // h)  # 2 for K and V
    label = 'MHA' if G == h else ('MQA' if G == 1 else f'GQA(G={G})')
    print(f'{label:12s}  params={params:,}  KV cache={kv_cache_elements:,} elements')

### Validate: GQA shrinks the KV cache

Multi-head attention stores $K$ and $V$ for every head; grouped-query attention shares them
across groups, so the KV cache — the memory bottleneck at long context — shrinks proportionally.
We confirm the cache sizes for MHA vs GQA vs MQA at a 8192-token context.

In [ ]:
n_ctx, h_, d_k_, b_ = 8192, 32, 128, 2
def kv_cache_bytes(G):
    return 2 * n_ctx * G * d_k_ * b_        # 2 = K and V; G = number of key/value heads
mha, gqa, mqa = kv_cache_bytes(h_), kv_cache_bytes(4), kv_cache_bytes(1)
print(f'KV cache/layer  MHA(G=32)={mha/1e6:.1f}MB  GQA(G=4)={gqa/1e6:.1f}MB  MQA(G=1)={mqa/1e6:.1f}MB')
assert gqa < mha and mqa < gqa, 'sharing key/value heads shrinks the KV cache (MHA > GQA > MQA)'
assert mha / gqa == h_ / 4, 'the cache shrinks exactly in proportion to the number of KV heads'
print('\n✅ GQA/MQA trade a little head diversity for a much smaller KV cache at long context')

## 5 — Visualizing attention patterns

Bidirectional self-attention vs causal attention on a short sentence.

In [ ]:
tokens = ['The', 'cat', 'sat', 'on', 'mat']
n = len(tokens)
torch.manual_seed(42)
X_demo = torch.randn(1, n, 32)
W_q2 = torch.randn(32, 8);  W_k2 = torch.randn(32, 8);  W_v2 = torch.randn(32, 8)
Q2 = X_demo[0] @ W_q2;  K2 = X_demo[0] @ W_k2;  V2 = X_demo[0] @ W_v2

mask_causal = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
_, W_bi  = scaled_dot_product_attention(Q2, K2, V2, mask=None)
_, W_cau = scaled_dot_product_attention(Q2, K2, V2, mask=mask_causal)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, W, title in zip(axes, [W_bi, W_cau], ['Bidirectional (encoder)', 'Causal (decoder)']):
    im = ax.imshow(W.detach().numpy(), cmap='viridis', vmin=0, vmax=1)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(tokens); ax.set_yticklabels(tokens)
    ax.set_title(title); ax.set_xlabel('attends to →')
    plt.colorbar(im, ax=ax)
plt.suptitle('Attention weights: row = query token, column = key token')
plt.tight_layout(); plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **missing $1/\sqrt{d_k}$** | softmax saturates, gradients vanish (demo) |
| **wrong mask** | leaking future tokens breaks autoregressive training (verified) |
| **KV cache at long context** | dominates memory; GQA/MQA shrink it (verified) |
| **too few KV heads (MQA)** | can lose quality vs MHA — GQA is the usual compromise |
| **fp precision** | attention softmax in fp16 needs care; use stable/flash kernels |

Demo: the $1/\sqrt{d_k}$ scaling keeps the attention distribution from collapsing.

In [ ]:
# Why the 1/sqrt(d_k) scaling matters: with large d_k the raw dot products have large variance,
# so softmax saturates onto one key — the attention distribution collapses (low entropy) and its
# gradient nearly vanishes. Dividing by sqrt(d_k) keeps scores in a sane range. We compare the
# entropy of the attention distribution with and without the scaling.
torch.manual_seed(0)
d_k = 128
Qb, Kb = torch.randn(4, d_k), torch.randn(4, d_k)
raw = Qb @ Kb.T
w_unscaled = F.softmax(raw, dim=-1)
w_scaled = F.softmax(raw / d_k**0.5, dim=-1)
ent = lambda w: -(w * torch.log(w + 1e-9)).sum(-1).mean().item()
print(f'attention entropy  unscaled={ent(w_unscaled):.3f}  scaled={ent(w_scaled):.3f}')
assert ent(w_scaled) > ent(w_unscaled), 'without 1/sqrt(d_k), softmax saturates (low entropy, vanishing gradient)'
print('\nLarge d_k inflates score variance -> softmax saturates. The 1/sqrt(d_k) factor prevents it.')

## ✏️ Your turn

**Task A — Cross-attention:** Implement cross-attention where queries come from a `decoder` tensor (shape `(B, n_dec, d)`) and keys/values come from an `encoder` tensor (shape `(B, n_enc, d)`). Show that the output has shape `(B, n_dec, d)` and that rows of the attention weight matrix each sum to 1.

**Task B — KV cache memory:** Given $n = 8192$ tokens, $h = 32$ heads, $d_k = 128$, bfloat16 (2 bytes per element), compute the KV cache size per layer for MHA, GQA ($G=4$), and MQA. How many 24 GB GPUs do you need to hold just the KV cache for a 40-layer model?

In [ ]:
# Task A: your cross-attention implementation
def cross_attention(x_dec, x_enc, W_q, W_k, W_v):
    # TODO(you): project x_dec → Q, x_enc → K,V; run scaled dot-product attention
    # No mask — decoder attends freely over all encoder positions
    ...

# Task B: KV cache arithmetic
n, h, d_k, bytes_per_elem = 8192, 32, 128, 2
for G, label in [(32, 'MHA'), (4, 'GQA G=4'), (1, 'MQA')]:
    cache_bytes = 2 * n * G * d_k * bytes_per_elem   # 2 = K and V
    # TODO(you): multiply by 40 layers, compare to 24 GB
    print(f'{label}: {cache_bytes / 1e6:.1f} MB per layer')

<details><summary>Solution — Task A</summary>

```python
def cross_attention(x_dec, x_enc, W_q, W_k, W_v):
    Q = x_dec @ W_q          # (B, n_dec, d_k)
    K = x_enc @ W_k          # (B, n_enc, d_k)
    V = x_enc @ W_v          # (B, n_enc, d_v)
    out, _ = scaled_dot_product_attention(Q, K, V, mask=None)
    return out               # (B, n_dec, d_v)

B, n_dec, n_enc, d = 2, 6, 10, 32
W_q = torch.randn(d, d); W_k = torch.randn(d, d); W_v = torch.randn(d, d)
enc = torch.randn(B, n_enc, d)
dec = torch.randn(B, n_dec, d)
print(cross_attention(dec, enc, W_q, W_k, W_v).shape)  # (2, 6, 32)
```
</details>

<details><summary>Solution — Task B</summary>

```python
n, h, d_k, bytes_per_elem, layers = 8192, 32, 128, 2, 40
gpu_gb = 24
for G, label in [(32,'MHA'), (4,'GQA G=4'), (1,'MQA')]:
    per_layer = 2 * n * G * d_k * bytes_per_elem
    total_gb = per_layer * layers / 1e9
    gpus = total_gb / gpu_gb
    print(f'{label}: {per_layer/1e6:.1f} MB/layer, {total_gb:.1f} GB total → {gpus:.1f}× A100')
# MHA:     2*8192*32*128*2 * 40 / 1e9 ≈ 42 GB → ~2 A100s just for KV cache!
# GQA G=4: ≈ 5 GB
# MQA:     ≈ 1.3 GB
```
</details>

## Key takeaways

- **Attention = softmax-weighted average of values:** weights are a distribution per query
  (verified).
- **Causal masking blocks the future:** upper-triangle weights are exactly 0 (verified) — the
  autoregressive constraint.
- **The $1/\sqrt{d_k}$ scaling** keeps softmax out of saturation; without it the distribution
  collapses (demo).
- **MHA → GQA → MQA** shrink the KV cache in proportion to KV heads (verified) — the long-context
  memory lever.